# Benchmark Result Analysis

This notebook reads only the saved benchmark JSON reports, validates the derived metrics, and generates the benchmark tradeoff plots used in the report discussion.

In [ ]:
from pathlib import Path
import json
import math
import subprocess

try:
    from IPython.display import Image, display
except ImportError:
    Image = None

    def display(value):
        print(value)

try:
    import pandas as pd
except ImportError:
    pd = None


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Code").exists() and (candidate / "Report").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


REPO_ROOT = find_repo_root()
REPORT_PATHS = {
    "cifar10": REPO_ROOT / "Code" / "results" / "benchmark" / "cifar10" / "forget_mixed" / "benchmark_report.json",
    "mufac": REPO_ROOT / "Code" / "results" / "benchmark" / "mufac" / "forget_mixed" / "benchmark_report.json",
}
ANALYSIS_DIR = REPO_ROOT / "Code" / "results" / "benchmark_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_ROWS_PATH = ANALYSIS_DIR / "benchmark_analysis_rows.json"
MAIN_FIGURE_PATH = REPO_ROOT / "Report" / "figures" / "benchmark-score-speedup.png"
DECOMPOSITION_FIGURE_PATH = ANALYSIS_DIR / "benchmark-score-decomposition.png"
EPSILON_FIGURE_PATH = ANALYSIS_DIR / "benchmark-epsilon-shares.png"
RENDERER_SCRIPT = REPO_ROOT / "Code" / "render_benchmark_analysis_plots.ps1"

DATASET_DISPLAY = {
    "cifar10": "CIFAR-10",
    "mufac": "MUFAC",
}
FAMILY_ORDER = [
    "baseline_train",
    "MSG",
    "FanchuanUnlearning",
    "SCRUB",
    "CT",
    "DELETE",
]
FAMILY_DISPLAY = {
    "baseline_train": "Train baseline",
    "MSG": "MSG",
    "FanchuanUnlearning": "Fanchuan",
    "SCRUB": "SCRUB",
    "CT": "CT",
    "DELETE": "DELETE",
}
EPSILON_LOW = 0.6931321804474443
EPSILON_HIGH = 50.0
EPSILON_TOL = 1e-5


def epsilon_share(epsilons: list[float], target: float) -> float:
    matches = [math.isclose(float(value), target, rel_tol=0.0, abs_tol=EPSILON_TOL) for value in epsilons]
    return sum(matches) / len(epsilons)


def build_analysis_rows(report_paths: dict[str, Path]) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    for dataset in DATASET_DISPLAY:
        report = json.loads(report_paths[dataset].read_text(encoding="utf-8"))
        reference_name = report["reference_family"]
        reference_summary = report["family_summaries"][reference_name]

        for family in FAMILY_ORDER:
            comparison = report["comparisons_to_reference"][family]
            summary = report["family_summaries"][family]
            epsilons = [float(value) for value in comparison["per_example_epsilons"]]
            retain_ratio = summary["retain_accuracy_mean"] / reference_summary["retain_accuracy_mean"]
            test_ratio = summary["test_accuracy_mean"] / reference_summary["test_accuracy_mean"]
            utility_product = retain_ratio * test_ratio
            speedup_vs_retrain = reference_summary["runtime_seconds_mean"] / summary["runtime_seconds_mean"]
            rows.append(
                {
                    "dataset": dataset,
                    "dataset_display": DATASET_DISPLAY[dataset],
                    "family": family,
                    "family_display": FAMILY_DISPLAY[family],
                    "forgetting_quality": comparison["forgetting_quality"],
                    "raw_final_score": comparison["raw_final_score"],
                    "final_score": comparison["final_score"],
                    "retain_accuracy_mean": summary["retain_accuracy_mean"],
                    "test_accuracy_mean": summary["test_accuracy_mean"],
                    "runtime_seconds_mean": summary["runtime_seconds_mean"],
                    "retain_ratio": retain_ratio,
                    "test_ratio": test_ratio,
                    "utility_product": utility_product,
                    "speedup_vs_retrain": speedup_vs_retrain,
                    "passed_efficiency_cutoff": comparison["passed_efficiency_cutoff"],
                    "epsilon_0p693_share": epsilon_share(epsilons, EPSILON_LOW),
                    "epsilon_50_share": epsilon_share(epsilons, EPSILON_HIGH),
                    "epsilon_count": len(epsilons),
                    "reference_runtime_seconds_mean": reference_summary["runtime_seconds_mean"],
                    "efficiency_ratio_threshold": comparison["efficiency_ratio_threshold"],
                }
            )
    return rows


def build_analysis_dataframe(rows: list[dict[str, object]]):
    if pd is None:
        return rows
    return pd.DataFrame(rows)


def display_analysis(rows: list[dict[str, object]]):
    columns = [
        "dataset_display",
        "family_display",
        "forgetting_quality",
        "raw_final_score",
        "final_score",
        "retain_accuracy_mean",
        "test_accuracy_mean",
        "runtime_seconds_mean",
        "retain_ratio",
        "test_ratio",
        "utility_product",
        "speedup_vs_retrain",
        "passed_efficiency_cutoff",
        "epsilon_0p693_share",
        "epsilon_50_share",
    ]
    if pd is not None:
        frame = pd.DataFrame(rows)[columns].copy()
        display(frame.round(4))
        return frame

    def format_value(value: object) -> str:
        if value is None:
            return "None"
        if isinstance(value, bool):
            return str(value)
        if isinstance(value, float):
            return f"{value:.4f}"
        return str(value)

    widths = {column: len(column) for column in columns}
    for row in rows:
        for column in columns:
            widths[column] = max(widths[column], len(format_value(row[column])))

    header = " | ".join(column.ljust(widths[column]) for column in columns)
    divider = "-+-".join("-" * widths[column] for column in columns)
    print(header)
    print(divider)
    for row in rows:
        print(" | ".join(format_value(row[column]).ljust(widths[column]) for column in columns))
    return rows


def validate_analysis_rows(rows: list[dict[str, object]]) -> None:
    expected_values = {
        ("cifar10", "MSG"): {"final_score": 0.26460519634507595, "speedup_vs_retrain": 7.9737113141840865},
        ("mufac", "MSG"): {"final_score": 0.27423745074807004, "speedup_vs_retrain": 9.38688903983151},
        ("cifar10", "FanchuanUnlearning"): {"final_score": 0.06666485929819888, "forgetting_quality": 0.06588888888888889},
        ("mufac", "FanchuanUnlearning"): {"final_score": 0.1139235984760837, "forgetting_quality": 0.11982968369829684},
        ("mufac", "SCRUB"): {"final_score": 0.1521758791604006, "test_accuracy_mean": 0.5, "retain_accuracy_mean": 0.6264767199444058},
    }

    for row in rows:
        assert math.isclose(
            row["raw_final_score"],
            row["forgetting_quality"] * row["utility_product"],
            rel_tol=1e-12,
            abs_tol=1e-12,
        )
        assert math.isclose(
            row["epsilon_0p693_share"] + row["epsilon_50_share"],
            1.0,
            rel_tol=0.0,
            abs_tol=1e-12,
        )
        cutoff_speedup = 1.0 / row["efficiency_ratio_threshold"]
        assert (row["speedup_vs_retrain"] > cutoff_speedup) == bool(row["passed_efficiency_cutoff"])

        if row["final_score"] is None:
            assert not row["passed_efficiency_cutoff"]
        else:
            assert row["passed_efficiency_cutoff"]
            assert math.isclose(row["final_score"], row["raw_final_score"], rel_tol=1e-12, abs_tol=1e-12)

    for key, metrics in expected_values.items():
        dataset, family = key
        row = next(item for item in rows if item["dataset"] == dataset and item["family"] == family)
        for metric_name, expected in metrics.items():
            assert math.isclose(row[metric_name], expected, rel_tol=1e-12, abs_tol=1e-12), (dataset, family, metric_name, row[metric_name], expected)


def render_plots(rows_path: Path) -> None:
    command = [
        "powershell.exe",
        "-NoProfile",
        "-ExecutionPolicy",
        "Bypass",
        "-File",
        str(RENDERER_SCRIPT),
        "-InputJson",
        str(rows_path),
        "-MainFigure",
        str(MAIN_FIGURE_PATH),
        "-DecompositionFigure",
        str(DECOMPOSITION_FIGURE_PATH),
        "-EpsilonFigure",
        str(EPSILON_FIGURE_PATH),
    ]
    subprocess.run(command, cwd=str(REPO_ROOT), check=True)


In [ ]:
analysis_rows = build_analysis_rows(REPORT_PATHS)
analysis_df = build_analysis_dataframe(analysis_rows)
display_analysis(analysis_rows)


In [ ]:
validate_analysis_rows(analysis_rows)
ANALYSIS_ROWS_PATH.write_text(json.dumps(analysis_rows, indent=2), encoding="utf-8")
print(f"Validated all derived metrics and wrote analysis rows to: {ANALYSIS_ROWS_PATH.relative_to(REPO_ROOT)}")


## Figures

The PowerShell renderer below creates the main report figure and two notebook-only supporting plots from the derived analysis rows.

In [ ]:
render_plots(ANALYSIS_ROWS_PATH)
print(f"Saved main report figure to: {MAIN_FIGURE_PATH.relative_to(REPO_ROOT)}")
print(f"Saved supporting figure to: {DECOMPOSITION_FIGURE_PATH.relative_to(REPO_ROOT)}")
print(f"Saved supporting figure to: {EPSILON_FIGURE_PATH.relative_to(REPO_ROOT)}")

if Image is not None:
    display(Image(filename=str(MAIN_FIGURE_PATH)))
    display(Image(filename=str(DECOMPOSITION_FIGURE_PATH)))
    display(Image(filename=str(EPSILON_FIGURE_PATH)))
